# 02 · Allocate prescribing to LSOAs and wards
Practice totals (01) are shared out to LSOAs by where each practice's registered patients live (paper eq. 2), then summed to 2018 wards.
Per-patient values = total ÷ registered patients (eq. 1). The logic is in `02_allocate_to_areas_workers.py`.

**Inputs:** `data/raw_data/prescriptions_epd/*`, `data/raw_data/gp_registrations/gp-reg-patients-all-lsoa.csv` (NHS Digital, April 2016), `data/raw_data/bnf_lists/opioids_OME.csv`, ONS LSOA 2011 → ward 2018 lookup.
**Outputs:** `data/prescriptions/{lsoa,ward}_prescriptions_2016_{condition}.csv`, `data/raw_data/prescriptions_epd/unallocated_2016.csv`, `extra_practices_2016.txt`.
**Runtime:** about 2 minutes. Run from this folder, after 01.

## 1. Registrations and the list of practices outside the region → SAVE

In [ ]:
import importlib
import pandas as pd

a = importlib.import_module('02_allocate_to_areas_workers')
YEAR = 2016
reg = a.load_registrations()
region = a.region_practices(YEAR)
extra = a.extra_practices(reg, region)
(a.EPD_DIR / f'extra_practices_{YEAR}.txt').write_text('\n'.join(extra))
lon = reg[reg.lsoa.isin(a.london_lookup().LSOA11CD)]
print(f'{reg.practice.nunique()} practices, {lon.patients.sum()/1e6:.2f} M London patients; '
      f'{lon[lon.practice.isin(region)].patients.sum()/lon.patients.sum():.4f} covered by the region pass, '
      f'{len(extra)} extra practices (≥ {a.MIN_EXTRA_PATIENTS} London patients) for 01 step 2')

## 2. Allocate every condition → SAVE

In [ ]:
report = a.run_year(YEAR)
report.round(4)

## Validation

In [ ]:
ward = pd.read_csv(a.OUT_DIR / f'ward_prescriptions_{YEAR}_total.csv')
print('wards:', len(ward), '| items per patient: median %.2f (10th percentile %.2f)' % (
    ward.Total_items_per_capita.median(), ward.Total_items_per_capita.quantile(.1)))
assert ward.WD18CD.is_unique